# Code mẫu — LoRA

LoRA giữ nguyên ma trận trọng số gốc, và chỉ học thêm một nhánh sửa. Nhánh sửa này là tích của hai ma trận nhỏ, nên có rất ít tham số. Các ví dụ dưới đây dùng những phép tuyến tính tính tay được để kiểm ba điều: nhánh sửa được cộng vào ra sao, gradient đi qua nhánh ấy thế nào, và rank giới hạn những gì nhánh sửa làm được.

## Ví dụ 1: Một hướng sửa, ba đầu vào, hai đầu ra

Ma trận gốc `W0` chỉ lấy hai tọa độ đầu của đầu vào. Adapter rank 1 gồm hai phần. `A` gom đầu vào thành một số duy nhất, là `x1 + 2x3`. `B` biến số ấy thành hai thay đổi trái dấu cho hai đầu ra. Nhánh sửa được nhân với hệ số `alpha / rank`, rồi cộng vào đầu ra gốc. Các trọng số đều đặt bằng tay để kiểm công thức.

In [ ]:
import numpy as np

W0 = np.array([[1., 0., 0.], [0., 1., 0.]])
A = np.array([[1., 0., 2.]])       # Shape (rank=1, số đầu vào=3).
B = np.array([[0.5], [-0.5]])     # Shape (số đầu ra=2, rank=1).
alpha, rank = 2, 1
scale = alpha / rank
X = np.array([[2., 1., 3.]])
base = X @ W0.T
update = scale * (X @ A.T) @ B.T
separate = base + update
merged = X @ (W0 + scale * B @ A).T
assert np.array_equal(separate, np.array([[10., -7.]]))
assert np.array_equal(separate, merged)
print("base:", base.tolist())
print("nhanh sua:", update.tolist())
print("output:", separate.tolist())
print("tham so full/adapter:", W0.size, A.size + B.size)
print("rank ma tran sua:", np.linalg.matrix_rank(B @ A))
print("merge trung:", np.array_equal(separate, merged))

base: [[2.0, 1.0]]
nhanh sua: [[8.0, -8.0]]
output: [[10.0, -7.0]]
tham so full/adapter: 6 5
rank ma tran sua: 1
merge trung: True


Với đầu vào `[2, 1, 3]`, nhánh gốc cho `[2, 1]`. `A` gom đầu vào thành $2+2\cdot3=8$. Nhân với `B` và hệ số 2, nhánh sửa là `[8, −8]`, nên đầu ra là `[10, −7]`. Ma trận sửa `B @ A` có rank 1, đúng như tên gọi.

Ta cũng có thể gộp nhánh sửa vào ma trận gốc trước, thành `W0 + scale · B @ A`, và kết quả y hệt. Đó là cách triển khai LoRA mà không làm chậm forward. Adapter ở đây có 5 số so với 6 số của ma trận đầy đủ, nên chỉ tiết kiệm đúng một số. Với ma trận lớn hàng nghìn chiều, khoản tiết kiệm mới thật sự đáng kể. Lưu ý là ma trận gốc vẫn phải nằm trong bộ nhớ để chạy forward. Số tham số học ít đi không có nghĩa tổng bộ nhớ ít đi tương ứng.

## Ví dụ 2: B bằng 0 thì bước đầu A đứng yên

Ta dùng hai đầu vào cơ sở và hai đầu ra hồi quy. `A` khởi tạo là `[1, 2]`, `B` khởi tạo bằng 0, và hệ số scale bằng 1. Vì `B` bằng 0, nhánh sửa ban đầu bằng 0, nên model lúc đầu y hệt model gốc. MSE lấy trung bình trên bốn phần tử. Ta xem gradient của `A` và `B` qua hai bước.

In [ ]:
import numpy as np
import torch
from torch import nn

torch.set_num_threads(1)
X = torch.eye(2, dtype=torch.float64)
Y = torch.tensor([[1., 2.], [3., 4.]], dtype=torch.float64)
W0 = nn.Parameter(torch.zeros((2, 2), dtype=torch.float64), requires_grad=False)
A = nn.Parameter(torch.tensor([[1., 2.]], dtype=torch.float64))
B = nn.Parameter(torch.zeros((2, 1), dtype=torch.float64))
optimizer = torch.optim.SGD([A, B], lr=0.1)
prediction = X @ W0.T + (X @ A.T) @ B.T
loss = ((prediction - Y)**2).mean()
loss.backward()
assert torch.equal(B.grad, torch.tensor([[-3.5], [-5.]], dtype=torch.float64))
print("buoc dau, A grad:", A.grad.tolist())
print("buoc dau, B grad:", B.grad.tolist())
print("base grad None:", W0.grad is None)
optimizer.step()
optimizer.zero_grad(set_to_none=True)
prediction = X @ W0.T + (X @ A.T) @ B.T
((prediction - Y)**2).mean().backward()
# Kiểm lại bằng chain rule viết tay. Mẫu số là 4, số phần tử của MSE.
G = 2 * (prediction.detach().numpy() - Y.numpy()) / Y.numel()
expected_A = B.detach().numpy().T @ G.T @ X.numpy()
assert np.allclose(A.grad.numpy(), expected_A)
print("buoc hai, A grad:", A.grad.numpy().round(5).tolist())

zero_A = nn.Parameter(torch.zeros_like(A))
zero_B = nn.Parameter(torch.zeros_like(B))
bad_prediction = (X @ zero_A.T) @ zero_B.T
((bad_prediction - Y)**2).mean().backward()
print("ca A/B khoi tao 0, gradient deu 0:",
      bool((zero_A.grad == 0).all() and (zero_B.grad == 0).all()))

buoc dau, A grad: [[0.0, 0.0]]
buoc dau, B grad: [[-3.5], [-5.0]]
base grad None: True
buoc hai, A grad: [[-0.48875, -1.1525]]
ca A/B khoi tao 0, gradient deu 0: True


Ở bước đầu, gradient của `A` bằng 0. Điều này đúng, vì gradient của `A` có nhân với `B`, mà `B` đang bằng 0. Gradient của `B` thì khác 0, nên `B` được cập nhật thành `[0.35, 0.5]`. Sang bước hai, `B` đã khác 0, và `A` bắt đầu nhận gradient. Ma trận gốc không được học, nên `.grad` của nó vẫn là `None`.

Phần cuối cho thấy vì sao không khởi tạo cả hai bằng 0. Khi đó gradient của `A` có nhân với `B`, gradient của `B` có nhân với `A`, nên cả hai cùng bằng 0 mãi. Adapter không bao giờ bắt đầu học được. Vì vậy LoRA khởi tạo `A` ngẫu nhiên và `B` bằng 0.

## Ví dụ 3: Rank 1 và rank 2 học cùng một thay đổi

Đây là bài hồi quy tổng hợp với sáu đầu vào và bốn đầu ra. Thay đổi cần học là một ma trận có rank đúng bằng 2: tăng đầu ra 1 theo đầu vào 1, và tăng đầu ra 2 theo đầu vào 2. Dữ liệu train và validation được sinh riêng. Hai adapter, rank 1 và rank 2, dùng chung ma trận gốc và cùng học 240 bước. Ta kiểm xem mỗi rank biểu diễn được tới đâu.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(44)
dtype = torch.float64
W0 = torch.arange(24, dtype=dtype).reshape(4, 6) / 50
delta = torch.zeros((4, 6), dtype=dtype)
delta[0, 0], delta[1, 1] = 1., 0.6
train_X = torch.randn((48, 6), dtype=dtype)
valid_X = torch.randn((40, 6), dtype=dtype)
train_Y = train_X @ (W0 + delta).T
valid_Y = valid_X @ (W0 + delta).T

class LoRALinear(nn.Module):
    def __init__(self, base, rank):
        super().__init__()
        self.base = nn.Parameter(base.clone(), requires_grad=False)
        self.A = nn.Parameter(torch.randn((rank, base.shape[1]), dtype=dtype) * 0.1)
        self.B = nn.Parameter(torch.zeros((base.shape[0], rank), dtype=dtype))
        self.scale = 2.0  # Đặt alpha = 2 × rank, nên alpha / rank luôn bằng 2.

    def forward(self, values):
        return values @ self.base.T + self.scale * (values @ self.A.T) @ self.B.T

def train_adapter(rank):
    torch.manual_seed(440)
    model = LoRALinear(W0, rank)
    optimizer = torch.optim.Adam([model.A, model.B], lr=0.04)
    model.train()
    for _ in range(240):
        optimizer.zero_grad(set_to_none=True)
        loss = ((model(train_X) - train_Y)**2).mean()
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.inference_mode():
        mse = ((model(valid_X) - valid_Y)**2).mean().item()
        merged_weight = model.base + model.scale * model.B @ model.A
        merged = valid_X @ merged_weight.T
        assert torch.allclose(model(valid_X), merged, atol=1e-10)
    assert torch.equal(model.base, W0) and model.base.grad is None
    trainable = model.A.numel() + model.B.numel()
    return mse, trainable

results = [train_adapter(rank) for rank in [1, 2]]
for rank, (mse, count) in zip([1, 2], results):
    print("rank:", rank, "trainable:", count, "valid MSE:", format(mse, ".3g"))
assert results[1][0] < results[0][0]
print("rank muc tieu:", int(torch.linalg.matrix_rank(delta)))
print("base giu nguyen, merge khop:", True)

rank: 1 trainable: 10 valid MSE: 0.0994
rank: 2 trainable: 20 valid MSE: 1.56e-12
rank muc tieu: 2
base giu nguyen, merge khop: True


Ma trận đầy đủ có 24 số. Rank 1 chỉ học 10 số, rank 2 học 20 số. Rank 1 chỉ biểu diễn được một hướng sửa, trong khi thay đổi cần học có hai hướng độc lập. Vì vậy MSE của nó dừng ở khoảng 0.0994, train thêm cũng không xuống được 0. Rank 2 đủ để biểu diễn chính xác thay đổi ấy, và MSE gần như bằng 0.

Rank đủ lớn chỉ có nghĩa là tồn tại một nghiệm tốt. Nó không bảo đảm optimizer sẽ tìm ra nghiệm đó. Hai dòng assert trong hàm còn kiểm rằng ma trận gốc không đổi, và bản đã gộp cho cùng kết quả. Đây là phép kiểm trên dữ liệu tuyến tính tổng hợp, nên chưa nói gì về chất lượng LoRA trên văn bản thật.